In [1]:
import pandas as pd
import numpy as np

In [2]:
path = "./raw_data.csv"

In [3]:
df = pd.read_csv(path)

In [4]:
df.columns = (
    df.columns
      .str.strip()                         # remove leading/trailing spaces
      .str.replace("umber Pets", "Number Pets")
      .str.replace(" ", "_")
)


In [5]:

text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )



In [6]:
binary_map = {
    "Yes": 1, "No": 0,
    "Y": 1, "N": 0,
    "True": 1, "False": 0
}

binary_cols = [
    "Retired",
    "Home_Owner",
    "Active_Lifestyle",
    "Wireless_Internet",
    "Smart_Phone",
    "Twitter_Acct",
    "LinkedIn_Acct",
    "Facebook_Acct",
    "News_Subscriber",
    "Coupon_Redemption",
    "High_Value_Customer"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .map(binary_map)
        )


In [7]:
currency_cols = [
    "Household_Income",
    "Car_Value",
    "Monthly_Spend_ProductA",
    "Cumulative_Spend_ProductA",
    "Monthly_Spend_ProductB",
    "Cumulative_Spend_ProductB",
    "Monthly_Spend_ProductC",
    "Cumulative_Spend_ProductC",
    "Total_Avg_Monthly_Spend"
]

for col in currency_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[$,]", "", regex=True)
        )

numeric_cols = currency_cols + [
    "Age",
    "Education_Years",
    "Employment_Years",
    "Household_Size",
    "Number_Pets"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [8]:

critical_fields = [
    "Age",
    "Household_Income",
    "Total_Avg_Monthly_Spend"
]

df = df.dropna(subset=critical_fields)


In [9]:
df = df[
    (df["Education_Years"] >= 0) &
    (df["Employment_Years"] >= 0) &
    (df["Education_Years"] < df["Age"]) &
    (df["Employment_Years"] <= (df["Age"] - 14))
]

In [10]:
df = df[
    (df["Household_Income"] >= 0) &
    (df["Car_Value"].isna() | (df["Car_Value"] >= 1000))
]


In [11]:
df = df[
    (df["Cumulative_Spend_ProductA"] >= df["Monthly_Spend_ProductA"]) &
    (df["Cumulative_Spend_ProductB"] >= df["Monthly_Spend_ProductB"]) &
    (df["Cumulative_Spend_ProductC"] >= df["Monthly_Spend_ProductC"])
]

In [12]:
df = df.drop_duplicates(subset="CustomerID")


In [13]:
print("Final Shape:", df.shape)
df.dtypes

Final Shape: (4137, 40)


CustomerID                    object
Region                         int64
Gender                        object
Age                            int64
Education_Years                int64
Employment_Years               int64
Job_Category                  object
Retired                        int64
Marital_Status                object
Household_Size                 int64
Household_Income             float64
NNumber_Pets                   int64
Home_Owner                   float64
Car_Ownership                 object
Car_Brand                     object
Car_Value                      int64
Commute_Distance              object
Political_Party               object
Votes                         object
Credit_Card                   object
Credit_Card_Tenure             int64
Active_Lifestyle               int64
TV_Watching_Hours              int64
Streaming_Svcs                object
Wireless_Internet              int64
Smart_Phone                    int64
Twitter_Acct                   int64
L

In [14]:
df.to_csv("customer_data_cleaned.csv", index=False)